# Evaluating Qwen3-8B on CAR-bench

This notebook implements the evaluation pipeline for the **Qwen3-8B-Instruct** model on the **CAR-bench** dataset. CAR-bench is a benchmark designed to assess in-car voice assistants on action-oriented tasks, handling user inputs, system context, policies, hallucination detection, and interactive disambiguation.

To run efficiently in cloud environments like Google Colab (T4 GPUs) or Kaggle:
- We deploy an optimized 4-bit quantized version: `unsloth/Qwen3-8B-bnb-4bit` (with `Qwen/Qwen3-8B-AWQ` as a fallback option) using the **vLLM** inference engine.
- We implement strict memory management controls to avoid Out-Of-Memory (OOM) errors.
- We establish an automated checkpoint-merging state recovery mechanism. If the Colab kernel restarts or disconnects, re-running the notebook will resume execution exactly where it left off, avoiding duplicate execution and API calls.


## 1. Environment Setup

We determine our run environment (Google Colab, Kaggle, or local workspace), clone the official CAR-bench repository, install package dependencies via `uv`, and append paths to `sys.path` to avoid `ModuleNotFoundError` issues.


In [ ]:
import os
import sys
import shutil

# Detect execution environment
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# If running locally, check if we need to navigate to the repo root
if not IN_COLAB:
    current_dir = os.getcwd().replace('\\', '/')
    if 'notebooks/car-bench-ijcai' in current_dir:
        print("Notebook is running inside the duplicate nested clone. Navigating out...")
        %cd ../..
    elif current_dir.endswith('/notebooks'):
        print("Notebook is in the notebooks directory. Navigating to workspace root...")
        %cd ..

# Clone repository if running in a clean cloud VM
if IN_COLAB or not os.path.exists("src"):
    print("Cloning repository to retrieve wiki.md policy and scripts...")
    !git clone --recursive https://github.com/CAR-bench/car-bench-ijcai.git
    %cd car-bench-ijcai
else:
    print("Running in local workspace.")
    # Clean up duplicate nested clone if present
    if os.path.exists("notebooks/car-bench-ijcai"):
        print("Cleaning up duplicate nested clone at notebooks/car-bench-ijcai...")
        import stat
        def remove_readonly(func, path, _):
            os.chmod(path, stat.S_IWRITE)
            func(path)
        try:
            shutil.rmtree("notebooks/car-bench-ijcai", onerror=remove_readonly)
            print("Successfully cleaned up duplicate clone.")
        except Exception as e:
            print(f"Could not clean up notebooks/car-bench-ijcai: {e}")

# Robust check/fix for submodule/dependency content in third_party/car-bench
if not os.path.exists("third_party/car-bench/pyproject.toml"):
    print("Submodule files missing in third_party/car-bench. Attempting to restore...")
    try:
        import subprocess
        subprocess.run(["git", "submodule", "update", "--init", "--recursive"], check=True)
    except Exception:
        pass
    if not os.path.exists("third_party/car-bench/pyproject.toml"):
        print("Directly cloning official car-bench dependency...")
        if os.path.exists("third_party/car-bench"):
            import stat
            def remove_readonly(func, path, _):
                os.chmod(path, stat.S_IWRITE)
                func(path)
            try:
                shutil.rmtree("third_party/car-bench", onerror=remove_readonly)
            except Exception:
                try:
                    subprocess.run(["rm", "-rf", "third_party/car-bench"], shell=True)
                except Exception:
                    pass
        if os.name == 'nt':
            print("Windows detected. Using sparse checkout with NTFS protection disabled to avoid colon in path errors...")
            try:
                import subprocess
                subprocess.run(["git", "-c", "core.protectNTFS=false", "clone", "--depth", "1", "--no-checkout", "https://github.com/CAR-bench/car-bench.git", "third_party/car-bench"], check=True)
                cwd_cb = os.path.abspath("third_party/car-bench")
                subprocess.run(["git", "sparse-checkout", "init", "--cone"], cwd=cwd_cb, check=True)
                subprocess.run(["git", "sparse-checkout", "set", "car_bench", "setup.py", "pyproject.toml", "docs", "templates"], cwd=cwd_cb, check=True)
                subprocess.run(["git", "-c", "core.protectNTFS=false", "checkout"], cwd=cwd_cb, check=True)
                print("Sparse checkout completed successfully.")
            except Exception as e:
                print(f"Windows sparse checkout failed: {e}")
        else:
            !git clone --depth 1 https://github.com/CAR-bench/car-bench.git third_party/car-bench

# Programmatically patch setup.py to use encoding='utf-8' to prevent UnicodeDecodeError on Windows
setup_py_path = "third_party/car-bench/setup.py"
if os.path.exists(setup_py_path):
    print("Patching third_party/car-bench/setup.py to handle UTF-8 encoding on Windows...")
    try:
        with open(setup_py_path, "r", encoding="utf-8") as f:
            setup_content = f.read()
        target_str = 'long_description=open("README.md").read(),'
        replacement_str = 'long_description=open("README.md", encoding="utf-8").read(),'
        if target_str in setup_content:
            setup_content = setup_content.replace(target_str, replacement_str)
            with open(setup_py_path, "w", encoding="utf-8") as f:
                f.write(setup_content)
            print("Successfully patched setup.py.")
        else:
            print("setup.py already patched or target string not found.")
    except Exception as e:
        print(f"Failed to patch setup.py: {e}")


In [ ]:
# Detect Python runtime environment
import sys
import os
import subprocess

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB or os.path.exists("/kaggle/working"):
    python_path = sys.executable
else:
    python_path = os.path.abspath(".venv/bin/python")
    if os.name == 'nt':
        python_path = os.path.abspath(".venv/Scripts/python.exe")
    if not os.path.exists(python_path):
        python_path = sys.executable

# Install uv package manager if not available
try:
    subprocess.run(["uv", "--version"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
    print("uv is already installed.")
except (subprocess.SubprocessError, FileNotFoundError):
    print("uv not found. Installing uv to user space...")
    !{sys.executable} -m pip install -q --user uv

# Install package dependencies (targeting the correct environment)
python_target = f'--python "{python_path}"' if not (IN_COLAB or os.path.exists("/kaggle/working")) else "--system"
print(f"Installing dependencies into Python environment: {python_path}")
!uv pip install {python_target} -q "a2a-sdk[http-server]>=1.0.0" httpx loguru pydantic python-dotenv uvicorn nest-asyncio matplotlib pandas seaborn psutil datasets litellm
!uv pip install {python_target} -e third_party/car-bench


## 2. Resource Monitoring & Memory Purging

To prevent GPU Out-of-Memory (OOM) errors and host RAM overflow during model loading and evaluation, we define helper functions to log usage statistics and purge PyTorch CUDA allocation caches.


In [ ]:
import gc
    import torch
import psutil
import socket
import os
import subprocess
# Dummy GPU operation to register active GPU usage with Colab resource monitor
else:

def is_port_in_use(port):
    """Checks if a port is in use locally."""
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        return s.connect_ex(('127.0.0.1', port)) == 0

def cleanup_ports(ports=[8080, 8081]):
    """Kills processes on target ports safely and quickly, protecting the notebook session."""
    my_pid = os.getpid()
    my_ppid = os.getppid()
    protected_pids = {my_pid, my_ppid}
    
    print(f"Scanning for active processes on ports: {ports}...")
    for port in ports:
        if not is_port_in_use(port):
            continue
        print(f"Port {port} is in use. Identifying process...")
        pids_to_kill = set()
        
        # Identify PIDs using target port using OS command (much faster than iterating all processes)
        try:
            if os.name == 'nt':
                cmd = f"netstat -ano | findstr :{port}"
                try:
                    output = subprocess.check_output(cmd, shell=True).decode()
                    for line in output.strip().split('\n'):
                        parts = line.strip().split()
                        if len(parts) >= 5:
                            # Usually netstat output format: Proto LocalAddress ForeignAddress State PID
                            # If state is not present, PID could be at index 4, otherwise at index 4 or 5
                            pid_candidate = parts[-1]
                            try:
                                pid = int(pid_candidate)
                                if pid not in protected_pids:
                                    pids_to_kill.add(pid)
                            except ValueError:
                                pass
                except subprocess.CalledProcessError:
                    pass
            else:
                try:
                    pids_str = subprocess.check_output(["lsof", "-ti", f":{port}"]).decode().split()
                    for pid_str in pids_str:
                        try:
                            pid = int(pid_str)
                            if pid not in protected_pids:
                                pids_to_kill.add(pid)
                        except ValueError:
                            pass
                except (subprocess.CalledProcessError, FileNotFoundError):
                    pass
        except Exception as e:
            print(f"Error identifying process for port {port}: {e}")
            
        # Fallback to psutil only for specific connections if OS tools failed
        if not pids_to_kill:
            try:
                for conn in psutil.net_connections(kind='inet'):
                    if conn.laddr.port == port and conn.pid:
                        if conn.pid not in protected_pids:
                            pids_to_kill.add(conn.pid)
            except Exception:
                pass
                
        # Filter identified PIDs to ensure safety before terminating
        for pid in list(pids_to_kill):
            try:
                proc = psutil.Process(pid)
                cmdline = proc.cmdline()
                cmd_str = " ".join(cmdline).lower()
                name = proc.name().lower()
                
                # Double check to protect Jupyter/Notebook itself
                if any(k in name or k in cmd_str for k in ["jupyter", "ipykernel", "colab"]):
                    print(f"Skipping protected PID {pid} ({proc.name()}) on port {port}.")
                    pids_to_kill.discard(pid)
                    continue
                    
                # Ensure it matches expected safe keywords to avoid killing system/user services
                is_safe_to_kill = any(kw in cmd_str or kw in name for kw in ["vllm", "server.py", "track_1", "evaluator", "car-bench", "ollama"])
                if not is_safe_to_kill:
                    print(f"Skipping PID {pid} ({proc.name()}) on port {port} (does not match safe keywords).")
                    pids_to_kill.discard(pid)
            except Exception:
                pids_to_kill.discard(pid)

        # Execute termination on safely identified PIDs
        for pid in pids_to_kill:
            try:
                proc = psutil.Process(pid)
                print(f"Terminating safe process {proc.name()} (PID {pid}) on port {port}...")
                proc.terminate()
                try:
                    proc.wait(timeout=5)
                    print(f"Process {pid} terminated successfully.")
                except psutil.TimeoutExpired:
                    print(f"Process {pid} did not exit. Force killing...")
                    proc.kill()
                    proc.wait(timeout=2)
            except Exception as e:
                print(f"Failed to terminate PID {pid}: {e}")

cleanup_ports()
purge_memory()
print_resource_usage()


## 3. Persistent Checkpointing & Checkpoint Merging

We mount Google Drive (if running on Colab) or configure a persistent output folder. We then implement a robust checkpoint-merging utility. 
This script reads the target output JSON (`qwen3_8b_benchmark.json`), checks which tasks have already completed successfully, and allows the run loop to skip them, preventing duplicate executions and minimizing API usage.


In [ ]:
PERSISTENT_DIR = "./output"
if IN_COLAB:
    try:
        from google.colab import drive
        print("Mounting Google Drive for persistent artifact backups...")
        drive.mount('/content/drive')
        PERSISTENT_DIR = "/content/drive/MyDrive/car_bench_eval_qwen3"
    except Exception as e:
        print(f"Google Drive mount failed: {e}. Output will be saved locally.")
elif os.path.exists("/kaggle/working"):
    PERSISTENT_DIR = "/kaggle/working/output"

os.makedirs(PERSISTENT_DIR, exist_ok=True)
print(f"Target persistent directory: {PERSISTENT_DIR}")

In [ ]:
import json
from datasets import load_dataset

# Configuration for evaluation splits (5 tasks per split for benchmark)
target_splits = {
    "base": 5,
    "hallucination": 5,
    "disambiguation": 5
}

def get_all_task_ids():
    """Retrieves standard task IDs from the Hugging Face dataset."""
    all_task_ids = {}
    for task_type, limit in target_splits.items():
        try:
            config_name = f"tasks_{task_type}"
            ds = load_dataset("johanneskirmayr/car-bench-dataset", config_name, split="train")
            all_task_ids[task_type] = [ds[i]["task_id"] for i in range(min(limit, len(ds)))]
        except Exception as e:
            print(f"Dataset download skipped or failed for {task_type}: {e}. Using fallback IDs.")
            all_task_ids[task_type] = [f"{task_type}_{i}" for i in range(limit)]
    return all_task_ids

def merge_and_save_results(main_file_path, temp_file_path):
    """Merges new trial runs into the main persistent benchmark JSON file."""
    if not os.path.exists(temp_file_path):
        print("No temporary results file found to merge.")
        return
    
    try:
        with open(temp_file_path, "r") as f:
            new_data = json.load(f)
    except Exception as e:
        print(f"Failed to read new results: {e}")
        return
        
    merged_data = {}
    if os.path.exists(main_file_path):
        try:
            with open(main_file_path, "r") as f:
                merged_data = json.load(f)
        except Exception as e:
            print(f"Failed to read existing checkpoint: {e}. Starting fresh.")
            
    if not merged_data:
        merged_data = {
            "metadata": new_data.get("metadata", {}),
            "summary_text": "",
            "summary": {},
            "final_result": {
                "score": 0.0,
                "max_score": 0,
                "pass_rate": 0.0,
                "task_rewards_by_split": {
                    "base": {},
                    "hallucination": {},
                    "disambiguation": {}
                },
                "pass_power_k_scores_by_split": {},
                "pass_at_k_scores_by_split": {},
                "detailed_results_by_split": {
                    "base": [],
                    "hallucination": [],
                    "disambiguation": []
                }
            },
            "results": [],
            "artifacts": []
        }
        
    # Merge results list
    existing_results = {r.get("task_id"): r for r in merged_data.get("results", []) if r.get("task_id")}
    for r in new_data.get("results", []):
        if r.get("task_id"):
            existing_results[r["task_id"]] = r
    merged_data["results"] = list(existing_results.values())
    
    # Merge detailed split results
    detailed_merged = merged_data["final_result"].setdefault("detailed_results_by_split", {
        "base": [], "hallucination": [], "disambiguation": []
    })
    detailed_new = new_data.get("final_result", {}).get("detailed_results_by_split", {})
    for split in ["base", "hallucination", "disambiguation"]:
        tasks_dict = {t.get("task_id"): t for t in detailed_merged.get(split, []) if t.get("task_id")}
        for t in detailed_new.get(split, []):
            if t.get("task_id"):
                tasks_dict[t["task_id"]] = t
        detailed_merged[split] = list(tasks_dict.values())
        
    # Merge task rewards
    rewards_merged = merged_data["final_result"].setdefault("task_rewards_by_split", {
        "base": {}, "hallucination": {}, "disambiguation": {}
    })
    rewards_new = new_data.get("final_result", {}).get("task_rewards_by_split", {})
    for split in ["base", "hallucination", "disambiguation"]:
        rewards_merged.setdefault(split, {}).update(rewards_new.get(split, {}))
        
    # Recompute metrics
    all_completed = []
    for split in ["base", "hallucination", "disambiguation"]:
        all_completed.extend(detailed_merged.get(split, []))
        
    total_reward = sum(t.get("reward", 0.0) for t in all_completed)
    num_completed = len(all_completed)
    pass_rate = (total_reward / num_completed * 100) if num_completed > 0 else 0.0
    
    merged_data["final_result"]["score"] = total_reward
    merged_data["final_result"]["max_score"] = num_completed
    merged_data["final_result"]["pass_rate"] = pass_rate
    merged_data["final_result"]["pass_power_k_scores"] = {"Pass^1": pass_rate / 100.0}
    merged_data["final_result"]["pass_at_k_scores"] = {"Pass@1": pass_rate / 100.0}
    
    # Update split-level pass rate
    merged_data["final_result"]["pass_power_k_scores_by_split"] = {}
    merged_data["final_result"]["pass_at_k_scores_by_split"] = {}
    for split in ["base", "hallucination", "disambiguation"]:
        split_tasks = detailed_merged.get(split, [])
        if split_tasks:
            split_reward = sum(t.get("reward", 0.0) for t in split_tasks)
            split_rate = split_reward / len(split_tasks)
            merged_data["final_result"]["pass_power_k_scores_by_split"][split] = {"Pass^1": split_rate}
            merged_data["final_result"]["pass_at_k_scores_by_split"][split] = {"Pass@1": split_rate}
            
    # Rebuild the final summary text
    summary_parts = [
        "CAR-bench Evaluation Summary",
        f"Tasks Evaluated: {num_completed}",
        f"Overall Pass Rate: {pass_rate:.1f}% ({total_reward:.1f}/{num_completed})",
        "",
        "Pass Metrics:",
        f"  Pass^1 (Pass@1): {pass_rate:.1f}%",
        "",
        "Split Breakdown:"
    ]
    for split in ["base", "hallucination", "disambiguation"]:
        split_tasks = detailed_merged.get(split, [])
        if split_tasks:
            split_reward = sum(t.get("reward", 0.0) for t in split_tasks)
            split_rate = (split_reward / len(split_tasks) * 100)
            summary_parts.append(f"  - {split.capitalize()}: {split_rate:.1f}% ({split_reward:.1f}/{len(split_tasks)})")
            for t in sorted(split_tasks, key=lambda x: x.get("task_id", "")):
                summary_parts.append(f"    * Task {t['task_id']}: {'✓' if t.get('reward', 0.0) >= 0.99 else '✗'} ({t.get('reward', 0.0):.2f})")
                
    merged_data["summary_text"] = "\n".join(summary_parts)
    
    with open(main_file_path, "w") as f:
        json.dump(merged_data, f, indent=2)
    print(f"Checkpoint successfully updated at {main_file_path}")

## 4. Local Ollama Inference Server

We configure and run the local **Ollama** server offline. To load models from the custom directory, we set the `OLLAMA_MODELS` environment variable to point to `E:\Thesis\models\ollama`. The notebook will automatically start the Ollama server in the background if it is not already active.


In [ ]:
import os
import sys
import httpx
import time
import subprocess

# Set Ollama offline models directory
os.environ["OLLAMA_MODELS"] = r"E:\Thesis\models\ollama"
print(f"Ollama models directory set to: {os.environ['OLLAMA_MODELS']}")

VLLM_PORT = 11434
VLLM_MODEL = "qwen3:8b"
OLLAMA_MODEL = VLLM_MODEL

# Install Ollama in Colab/Kaggle if running there
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB or os.path.exists("/kaggle/working"):
    print("Cloud environment detected. Installing Ollama...")
    !curl -fsSL https://ollama.com/install.sh | sh

# Check if Ollama server is already running
print(f"Checking connection to Ollama local server on port {VLLM_PORT}...")
ollama_already_running = False
try:
    response = httpx.get(f"http://localhost:{VLLM_PORT}/v1/models", timeout=5)
    if response.status_code == 200:
        print("Ollama server is already running!")
        ollama_already_running = True
except Exception:
    pass

vllm_proc = None # Reusing variable name to maintain compatibility with other cells
if not ollama_already_running:
    print("Ollama server is not running. Spawning background Ollama serve process...")
    env = os.environ.copy()
    env["OLLAMA_MODELS"] = r"E:\Thesis\models\ollama"
    creationflags = 0
    if os.name == 'nt':
        creationflags = 0x08000000 # CREATE_NO_WINDOW
    
    try:
        vllm_proc = subprocess.Popen(
            ["ollama", "serve"],
            env=env,
            creationflags=creationflags,
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL
        )
        print("Spawned background Ollama server process.")
    except Exception as e:
        print(f"ERROR spawning Ollama process: {e}")
        print("Please ensure Ollama CLI is installed and in your system PATH.")

# Poll server until active
if not ollama_already_running:
    print("Waiting for Ollama server to respond...")
    is_ready = False
    for attempt in range(30):
        try:
            response = httpx.get(f"http://localhost:{VLLM_PORT}/v1/models", timeout=2)
            if response.status_code == 200:
                print("\nOllama server successfully started!")
                is_ready = True
                break
        except Exception:
            pass
        print(".", end="", flush=True)
        time.sleep(2)
    if not is_ready:
        print("\n[WARNING] Ollama server failed to start within the timeout period.")

# Verify available models
try:
    response = httpx.get(f"http://localhost:{VLLM_PORT}/v1/models", timeout=5)
    if response.status_code == 200:
        models = [m["id"] for m in response.json().get("data", [])]
        print(f"Available models in E:\\Thesis\\models\\ollama: {models}")
        if OLLAMA_MODEL in models:
            print(f"Model '{OLLAMA_MODEL}' is ready!")
        else:
            print(f"WARNING: Model '{OLLAMA_MODEL}' not found. Available models are: {models}")
    else:
        print(f"Ollama returned status code {response.status_code}")
except Exception as e:
    print(f"Could not retrieve models list: {e}")

vllm_already_running = ollama_already_running


## 5. Credentials Configuration

We request the user's Gemini API Key. The User Simulator (simulating the driver) and the Policy Evaluator (judging policy constraints) both utilize Gemini API models.


In [ ]:
import getpass

print("Enter your OpenAI API Key (used for simulating user/evaluating policies):")
openai_key = getpass.getpass()

# Write env configuration to .env file
with open(".env", "w") as f:
    f.write(f"OPENAI_API_KEY={openai_key}\n")
    # Redirect agent specifically using AGENT_API_BASE instead of breaking OpenAI globally
    f.write(f"AGENT_API_BASE=http://localhost:{VLLM_PORT}/v1\n")
    f.write("AGENT_API_KEY=ollama\n")
    f.write("OLLAMA_MODELS=E:\\Thesis\\models\\ollama\n")

print("Created .env configuration file successfully.")


## 6. Execution Loop with State Recovery

We check existing checkpoints, filter out completed tasks, write a temporary TOML scenario containing only the remaining tasks, and call `car-bench-run`. 
After the run is complete, we merge the new results back into the persistent file.


In [ ]:
import os
import sys

# Determine Python runtime executable path
if IN_COLAB or os.path.exists("/kaggle/working"):
    python_path = sys.executable
else:
    python_path = os.path.abspath(".venv/bin/python")
    if os.name == 'nt':
        python_path = os.path.abspath(".venv/Scripts/python.exe")
    if not os.path.exists(python_path):
        python_path = sys.executable

python_path_formatted = python_path.replace('\\', '/')
print(f"Using Python runtime: {python_path_formatted}")

# Configure path to main result file
main_result_file = os.path.join(PERSISTENT_DIR, "qwen3_8b_benchmark.json")

# Determine total target tasks and completed tasks
all_task_ids = get_all_task_ids()
total_target_tasks = sum(len(ids) for ids in all_task_ids.values())

completed_ids = set()
if os.path.exists(main_result_file):
    try:
        with open(main_result_file, "r") as f:
            old_data = json.load(f)
        detailed = old_data.get("final_result", {}).get("detailed_results_by_split", {})
        for split, tasks in detailed.items():
            for t in tasks:
                if "task_id" in t:
                    completed_ids.add(t["task_id"])
    except Exception as e:
        print(f"Failed to read existing checkpoint results: {e}. Starting fresh.")

print(f"Progress: {len(completed_ids)}/{total_target_tasks} tasks completed.")

# Check for remaining tasks
remaining_tasks = {}
for split, task_ids in all_task_ids.items():
    remaining_tasks[split] = [tid for tid in task_ids if tid not in completed_ids]
    print(f"  {split.capitalize()} split: {len(remaining_tasks[split])} tasks remaining.")

total_remaining = sum(len(ids) for ids in remaining_tasks.values())

if total_remaining == 0:
    print("All tasks have already been completed successfully!")
else:
    # Generate scenario TOML string
    lines = [
        "# Dynamic CAR-bench scenario config for Qwen3-8B-Instruct",
        "[evaluator]",
        'endpoint = "http://127.0.0.1:8081"',
        f'cmd = "{python_path_formatted} src/evaluator/server.py --host 127.0.0.1 --port 8081"',
        "",
        "[agent_under_test]",
        'endpoint = "http://127.0.0.1:8080"',
        f'cmd = "{python_path_formatted} src/track_1_agent_under_test/server.py --host 127.0.0.1 --port 8080 --agent-llm openai/{VLLM_MODEL} --temperature 0.0"',
        "",
        "[config]",
        "num_trials = 1",
        'task_split = "train"',
        "max_steps = 30",
        'user_model = "openai/gpt-4o-mini"',
        'user_provider = "openai"',
        'policy_evaluator_model = "openai/gpt-4o-mini"',
        'policy_evaluator_provider = "openai"'
    ]
    
    for split, task_ids in remaining_tasks.items():
        if task_ids:
            lines.append(f"tasks_{split}_task_id_filter = {json.dumps(task_ids)}")
            
    temp_toml_path = "scenarios/track_1_agent_under_test/qwen3_temp_scenario.toml"
    os.makedirs(os.path.dirname(temp_toml_path), exist_ok=True)
    with open(temp_toml_path, "w") as f:
        f.write("\n".join(lines))
        
    print(f"Generated scenario TOML configuration for {total_remaining} remaining tasks.")
    temp_output_path = "output/qwen3_temp_output.json"
    os.makedirs("output", exist_ok=True)
    if os.path.exists(temp_output_path):

### 6.1 Run Evaluation CLI

We clean up any leftover agent/evaluator ports and execute the benchmark run using the official `uv run car-bench-run` tool.


In [ ]:
# Clean up leftover agent/evaluator ports before running
cleanup_ports([8080, 8081])

# Run the benchmark scenario using the official BTC syntax
!uv run car-bench-run scenarios/track_1_agent_under_test/qwen3_temp_scenario.toml --output output/qwen3_temp_output.json --show-logs

### 6.2 Checkpoint Consolidation and Memory Cleanup

We load the newly generated results, merge them into our main checkpoint file, and purge the system and GPU memory.


In [ ]:
# Merge results and cleanup memory
if os.path.exists("output/qwen3_temp_output.json"):
    merge_and_save_results(main_result_file, "output/qwen3_temp_output.json")
else:
    print("No new evaluation output found to merge.")

purge_memory()

## 7. Results Visualization and Comparison

We load our final consolidated results and compare them directly against Qwen2.5-Coder-7B-Instruct (local baseline) and other models on the CAR-bench leaderboard.


In [ ]:
import os
import json
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from IPython.display import display, HTML

# Locate main consolidated results file
main_result_file = os.path.join(PERSISTENT_DIR, "qwen3_8b_benchmark.json")

if not os.path.exists(main_result_file) and os.path.exists("output/qwen3_temp_output.json"):
    main_result_file = "output/qwen3_temp_output.json"

try:
    with open(main_result_file, "r") as f:
        data = json.load(f)
        
    final_result = data.get("final_result", {})
    pass_rate = final_result.get("pass_rate", 0.0)
    rewards_by_split = final_result.get("pass_power_k_scores_by_split", {})
    
    # Extract split-level scores
    splits = ["base", "hallucination", "disambiguation"]
    split_scores = []
    for split in splits:
        split_score = rewards_by_split.get(split, {}).get("Pass^1", 0.0) * 100
        split_scores.append(split_score)
        
    print("=== FINAL CONSOLIDATED SCORES ===")
    print(f"Overall Pass Rate: {pass_rate:.1f}%")
    print(f"Base Split Pass Rate: {split_scores[0]:.1f}%")
    print(f"Hallucination Split Pass Rate: {split_scores[1]:.1f}%")
    print(f"Disambiguation Split Pass Rate: {split_scores[2]:.1f}%")
    
    # Plot results
    sns.set_theme(style="whitegrid")
    plt.figure(figsize=(10, 6))
    categories = ["Overall", "Base Split", "Hallucination Split", "Disambiguation Split"]
    values = [pass_rate] + split_scores
    
    ax = sns.barplot(x=categories, y=values, palette="viridis")
    plt.title("CAR-bench Scores (Qwen3-8B-Instruct)", fontsize=14, fontweight="bold", pad=15)
    plt.ylabel("Pass Rate (%)", fontsize=12)
    plt.ylim(0, 100)
    
    for p in ax.patches:
        ax.annotate(f"{p.get_height():.1f}%", (p.get_x() + p.get_width() / 2., p.get_height() + 1.5),
                    ha='center', va='center', fontsize=11, fontweight="semibold", color='black', xytext=(0, 5),
                    textcoords='offset points')
                    
    plt.tight_layout()
    chart_path = os.path.join(PERSISTENT_DIR, "qwen3_scores.png")
    plt.savefig("output/qwen3_scores.png", dpi=300)
    plt.savefig(chart_path, dpi=300)
    plt.show()
    
    # Leaderboard comparison
    comparison_data = {
        "Model": [
            "Claude Opus 4.6",
            "GPT-5",
            "Gemini 2.5 Pro",
            "Qwen3-8B-Instruct (This Run)",
            "Qwen2.5-Coder-7B-Instruct (Baseline)",
            "Qwen3-32B",
            "xLAM-2-32B"
        ],
        "Alignment Method": [
            "RLHF",
            "RLHF",
            "RLHF",
            "Raw SFT/RLHF",
            "Raw (Unaligned)",
            "Base SFT/RLHF",
            "SFT/DPO"
        ],
        "Overall Pass Rate": [
            "58.0%",
            "54.0%",
            "38.0%",
            f"{pass_rate:.1f}%",
            "20.0%",
            "31.0%",
            "16.0%"
        ]
    }
    
    df = pd.DataFrame(comparison_data)
    display(HTML("<h3>CAR-bench Leaderboard Comparison</h3>"))
    display(df)
    
except Exception as e:
    print(f"Visualization error: {e}. Ensure that results are available and correctly formatted.")

## 8. Graceful Shutdown & Resource Cleanup

To free up all host resources, we explicitly terminate the spawned Ollama server subprocess.


In [ ]:
# Cleanup cell to save disk space and clean memory
import os
import shutil
import gc
try:
    import torch
    HAS_TORCH = True
except ImportError:
    HAS_TORCH = False

# Terminate the background Ollama process if we spawned it
if 'vllm_proc' in globals() and vllm_proc is not None:
    print("Terminating background Ollama server...")
    vllm_proc.terminate()
    try:
        vllm_proc.wait(timeout=5)
        print("Ollama server terminated.")
    except Exception:
        vllm_proc.kill()
        print("Ollama server force killed.")

print("Cleaning up temporary files...")
temp_files = [
    "vllm_server.log",
    "scenarios/track_1_agent_under_test/qwen_coder_baseline.toml",
    "scenarios/track_1_agent_under_test/qwen3_temp_scenario.toml",
    "output/qwen3_temp_output.json"
]
for f in temp_files:
    if os.path.exists(f):
        try:
            os.remove(f)
            print(f"  Removed: {f}")
        except Exception as e:
            print(f"  Could not remove {f}: {e}")

print("Purging PyTorch CUDA memory cache...")
gc.collect()
if HAS_TORCH and torch.cuda.is_available():
    if HAS_TORCH: torch.cuda.empty_cache()
    if HAS_TORCH: torch.cuda.ipc_collect()

print("Purging pip download cache...")
!pip cache purge
print("Cleanup complete!")
